# 🏛️ MEF · Canon y sobrecanon petrolero recaudado por cada municipalidad (2005–hoy)

PERUPETRO publica el canon **por municipalidad** solo en 1993–2002 (Excel de Distribución) y el año en curso (Excel de Canon Municipalidades). Desde 2003 entrega el total municipal a un intermediario (CND hasta 2006; PCM desde 2007), que hace el reparto. Este notebook completa los años que faltan con el **MEF – Consulta Amigable de Ingresos** (`apps5.mineco.gob.pe/transparenciaingresos/mensual/`), monto **recaudado** por cada municipalidad.

**Cómo lee la consulta:** la consulta de ingresos funciona con botones: elegir una fila y pulsar la dimensión siguiente envía el formulario de la página (postback ASP.NET). Los filtros puestos en la URL, incluida la de «Exportar», se ignoran (probado con `Navegar_1` y `Navegar_6`, 2026). El notebook hace lo mismo que el navegador, con `requests` y sin navegador: abre `default.aspx?y=AAAA`, toma el marco `Navegar_N.aspx` del año y en cada paso envía el formulario con la fila elegida y el botón (Rubro, Tipo de Recurso, Nivel de Gobierno, Gob.Loc./Mancom., Departamento, Municipalidad). Cada paso se valida con el historial (breadcrumb) y la cabecera de la tabla. Los montos salen del valor exacto de cada fila (con céntimos).

- **Desde 2007**: fuente 5 (Recursos determinados) → rubro 18 → tipos «5: Canon y sobrecanon petrolero» y «K: Sub cuenta – …» (se suman) → gobiernos locales → departamento → municipalidad.
- **2005–2006**: fuente 01 (Canon y sobrecanon) → tipo de recurso; el tipo 4 petrolero trae poco a nivel municipal (Loreto 2005: S/ 0,91 millones; 2006: S/ 0,14 millones), y el grueso está en «0: NORMAL», que reúne **todos** los canones; se suman ambos, cada fila con su rótulo. Desde 2007 **no** se usa «0: CANON» como respaldo: en 2008 y 2009 ese tipo da a los gobiernos locales de Loreto apenas S/ 1,18 y S/ 0,19 millones; el canon petrolero está en los tipos 5 y K. Con `SI_NO_HAY_PETROLERO = "Usar NORMAL 2005–2006"` se toma NORMAL y queda rotulado «todos los canones» en el CSV y en el mapa.
- **2003–2004**: sin detalle por municipalidad (el canon petrolero municipal figura a nombre del Consejo Nacional de Descentralización).
- **Primero, la distribución por nivel de gobierno**: para cada año y tipo de recurso petrolero, lo recaudado por el gobierno nacional (E), los gobiernos locales (M) y los gobiernos regionales (R), en total país y por departamento → `DATA/PP_canon_nivel_gobierno_mef_anio.csv` (control: suma de niveles = total del tipo de recurso, en `CONTROL/CONTROL PP canon_nivel_gobierno_mef.csv`).
- **Después, el detalle por municipalidad** (nivel M). **Resultado:** `DATA/PP_canon_municipal_mef_anio.csv` y `CONTROL/CONTROL PP canon_municipal_mef.csv` (suma de municipalidades vs. total del departamento en el MEF, y vs. el total municipal de PERUPETRO: *recaudado* y *transferido* pueden diferir por fechas de registro).

- **Versión mensual** (`DETALLE = "Anual y mensual"`): con el botón «Mes» se baja lo mismo por mes → `DATA/PP_canon_municipal_mef_mes.csv` (AÑO, MES, municipalidad, tipo, recaudado) y `DATA/PP_canon_nivel_gobierno_mef_mes.csv`; control en `CONTROL/CONTROL PP canon_mef_mes.csv` (suma de los meses = total anual). Toma unas 12 veces más consultas que la versión anual.

**Bloque 1** importa las librerías. **Bloque 2** descarga y controla.

In [ ]:
#@title Bloque 1 · Librerías { display-mode: "form" }
import importlib
for m in ("requests", "bs4", "pandas"):
    importlib.import_module(m)
print("✅ Librerías listas")


In [ ]:
#@title Bloque 2 · Descargar y controlar { display-mode: "form" }
ANIO_INICIO = 2003  #@param {type:"integer"}
ANIO_FIN = 2026  #@param {type:"integer"}
SI_NO_HAY_PETROLERO = "Usar NORMAL 2005–2006 (todos los canones)"  #@param ["Usar NORMAL 2005–2006 (todos los canones)", "Dejar sin dato"]
DETALLE = "Anual"  #@param ["Anual", "Anual y mensual"]
DEPARTAMENTO = "16: LORETO"  #@param ["LOS 6 PETROLEROS", "16: LORETO", "25: UCAYALI", "20: PIURA", "24: TUMBES", "10: HUANUCO", "08: CUSCO"]
import os, re, unicodedata, pandas as pd
from datetime import date
DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
FUENTE = "MEF, Consulta Amigable de Ingresos (recaudado)"
ruta_local = "/content/pp_mef"; os.makedirs(ruta_local, exist_ok=True)

# ── Google Drive ──
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false", fields="nextPageToken, files(id,name)",
                               pageSize=1000, pageToken=token, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out
def subir(ruta_archivo, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_archivo)
    media = MediaFileUpload(ruta_archivo, mimetype=mime, resumable=True)
    if nombre in existentes:
        drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
    else:
        existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                                  fields="id", supportsAllDrives=True).execute()["id"]
def bajar(nombre, existentes):
    if nombre not in existentes:
        return None
    p = os.path.join(ruta_local, nombre)
    with open(p, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=existentes[nombre], supportsAllDrives=True)); hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()
    return pd.read_csv(p, encoding="utf-8-sig")
datos_drive, control_drive = archivos_en(DRIVE_FOLDER_ID), archivos_en(DRIVE_FOLDER_CONTROL_ID)

# ── Lector del MEF (Consulta Amigable de Ingresos): canon y sobrecanon petrolero por nivel de gobierno y por municipalidad ──
# La consulta de ingresos navega por FORMULARIO ASP.NET (postback), no por URL:
#   · los filtros puestos en la URL (también en la de «Exportar») se IGNORAN: probado a mano con Navegar_1 y Navegar_6 (2026),
#     la exportación con 10=5 devolvió el total del país;
#   · cada clic = POST al Navegar_N.aspx con los campos ocultos de la página (__VIEWSTATE, __EVENTVALIDATION…),
#     grp1 = valor del radio de la fila elegida («código/PIA/PIM/recaudado») y el botón de la dimensión siguiente
#     (p. ej. ctl00$CPH1$BtnRubro=Rubro). Es lo que hace el navegador; aquí se hace con requests, sin navegador;
#   · el estado viaja en el __VIEWSTATE: cada página leída se guarda y se vuelve a enviar con otra fila para ramificar
#     (otro departamento, otro nivel) sin repetir el camino. Cada paso se valida con el historial (breadcrumb) y la
#     cabecera de la tabla; si una página guardada deja de servir, se rehace el camino desde la raíz;
#   · la página de cada año se toma del marco de default.aspx?y=AAAA (<iframe> Navegar_N.aspx; en 2003, <frame> Navegar.aspx).
import re, time, random, unicodedata, html as _html
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

BASE = "https://apps5.mineco.gob.pe/transparenciaingresos/mensual"
D_FUENTE, D_RUBRO, D_TIPO, D_NIVEL = "Fuente de Financiamiento", "Rubro", "Tipo de Recurso", "Nivel de Gobierno"
D_GOBLOC, D_DEPTO, D_MUNI = "Gob.Loc./Mancom.", "Departamento", "Municipalidad"
D_SECTOR, D_PLIEGO, D_MES = "Sector", "Pliego", "Mes"
_MESES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO", "AGOSTO", "SETIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"]

def mes_num(cod, etiqueta=""):
    """«1» / «01: ENERO» / «SETIEMBRE» → 1..12 (None si no se reconoce)."""
    if str(cod).strip().isdigit() and 1 <= int(cod) <= 12:
        return int(cod)
    t = norm(etiqueta).upper().replace("SEPTIEMBRE", "SETIEMBRE")
    return next((i + 1 for i, m in enumerate(_MESES) if m in t), None)
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

def norm(t):
    t = unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t).strip().lower()

def a_numero(t):
    t = (t or "").replace(",", "").replace("\xa0", "").strip()
    try:
        return float(t)
    except ValueError:
        return None

def _texto(td):
    return " ".join(td.get_text(" ", strip=True).split())

def _abreviar(t):
    """«Gobiernos Locales / Mancomunidades» y «Gob.Loc./Mancom.» → «gob loc manc» (botón y cabecera usan nombres distintos)."""
    t = norm(t)
    if "manc" in t and "gob" in t:
        return "gob loc manc"
    return t

def solo_letras(t):
    """Para comparar la fila elegida con el historial sin comillas ni signos (los meses se muestran como «1: 'Enero'»)."""
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", norm(t))).strip()

def misma_dim(cabecera, dim):
    c, d = _abreviar(cabecera), _abreviar(dim)
    return bool(c) and (c.startswith(d[:10]) or d.startswith(c[:10]))

class Pagina:
    """Página del navegador. dim: dimensión listada · filtros: historial [(etiqueta, recaudado)] (el 1.º es TOTAL) ·
    filas: [(codigo, etiqueta, recaudado)] · radios: {codigo: valor del radio} · botones: {texto: nombre} ·
    ocultos: campos ocultos del formulario · accion: destino del formulario."""
    def __init__(self, contenido, url=""):
        self.url = url
        if isinstance(contenido, bytes):
            try:
                contenido = contenido.decode("utf-8")
            except UnicodeDecodeError:
                contenido = contenido.decode("cp1252", errors="replace")
        s = BeautifulSoup(contenido, "html.parser")
        f = s.find("form")
        self.accion = _html.unescape(f.get("action") or "") if f else ""
        # todos los campos que el navegador envía con el formulario: ocultos, de texto y listas (p. ej. el año, DrpYear)
        self.ocultos = {i.get("name"): i.get("value") or "" for i in s.find_all("input")
                        if i.get("name") and (i.get("type") or "text").lower() in ("hidden", "text")}
        self.anio = None
        for sel in s.find_all("select"):
            if not sel.get("name"):
                continue
            op = sel.find("option", selected=True) or sel.find("option")
            self.ocultos[sel.get("name")] = (op.get("value") if op and op.get("value") is not None else (op.get_text(strip=True) if op else ""))
            if "year" in sel.get("name").lower() or "anio" in sel.get("name").lower():
                self.anio = self.ocultos[sel.get("name")]
        self.botones = {" ".join((i.get("value") or "").split()): i.get("name")
                        for i in s.find_all("input", attrs={"type": "submit"}) if i.get("name")}
        data = s.find("table", class_="Data")
        self.tiene_tabla = data is not None
        for t in s(["script", "style"]):
            t.decompose()
        self.texto = " ".join(s.get_text(" ", strip=True).split())     # para el diagnóstico cuando no hay tabla
        mt = s.find("table", class_="MapTable")
        celdas = [_texto(td) for td in (mt.find("tr").find_all("td") if mt and mt.find("tr") else [])]
        self.dim = next((c for c in celdas if c and c not in ("PIA", "PIM", "Recaudado")), "")
        hist = s.find("table", class_="History")
        self.filtros = []
        for tr in (hist.find_all("tr") if hist else []):
            tds = [_texto(td) for td in tr.find_all("td")]
            et = next((t for t in tds if t and a_numero(t) is None), "")
            if et:
                self.filtros.append((et, a_numero(tds[-1])))
        self.filas, self.radios = [], {}
        for tr in (data.find_all("tr") if data else []):
            r = tr.find("input", attrs={"type": "radio"})
            tds = [_texto(td) for td in tr.find_all("td")]
            et = next((t for t in tds if t and a_numero(t) is None), "")
            if not et or r is None:
                continue
            partes = (r.get("value") or "").split("/")
            rec = a_numero(partes[-1]) if len(partes) >= 3 else None
            if rec is None:
                rec = a_numero(tds[-1]) or 0.0
            m = _CODIGO.match(et)
            cod = (partes[0] if partes and partes[0] else (m.group(1) if m else et)).strip()
            self.filas.append((cod, et, rec))
            self.radios[cod] = r.get("value") or ""

    def boton(self, dim):
        return next(((t, n) for t, n in self.botones.items() if misma_dim(t, dim)), None)

class MEFIngresos:
    def __init__(self, log=print, pausa=(0.8, 1.6), base=BASE):
        self.log, self.pausa, self.base = log, pausa, base.rstrip("/")
        self.s = requests.Session()
        self.s.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                                             "Chrome/124.0 Safari/537.36", "Accept-Language": "es-PE,es;q=0.9"})
        self.pagina = None                               # número N de Navegar_N.aspx del año abierto
        self._anio = None
        self._cache = {}                                 # (filtros…, dimensión listada) → Pagina

    def _pedir(self, metodo, url, datos=None, reintentos=3):
        err = ""
        for i in range(reintentos):
            try:
                r = self.s.request(metodo, url, data=datos, timeout=90,
                                   headers={"Referer": url if metodo == "POST" else f"{self.base}/default.aspx"})
                if r.status_code == 200:
                    time.sleep(random.uniform(*self.pausa))
                    return Pagina(r.content, r.url), None
                err = f"HTTP {r.status_code}"
            except requests.RequestException as e:
                err = f"{type(e).__name__}: {e}"
            if i + 1 < reintentos:
                time.sleep(min(20, 2 ** i + random.random()))
        return None, err

    def preparar(self, anio, forzar=False):
        """Abre el año: default.aspx?y=AAAA y el marco que trae (Navegar_N.aspx); la raíz debe listar las fuentes."""
        if self._anio == anio and not forzar:
            return True
        self._anio, self._cache = None, {}
        try:
            r = self.s.get(f"{self.base}/default.aspx", params={"y": anio}, timeout=60)
            # <iframe> en los años recientes; <frame src="Navegar.aspx?y=2003"> (sin número) en 2003
            m = re.search(r"<i?frame[^>]+src=[\"']([^\"']*Navegar(?:_\d+)?\.aspx[^\"']*)[\"']", r.text, re.I)
            src = _html.unescape(m.group(1)) if m else f"Navegar_6.aspx?y={anio}"
        except requests.RequestException as e:
            self.log(f"   ❌ default.aspx {anio}: {e}"); return False
        # el marco que indica default.aspx y, si no sirve, los demás Navegar_N del año
        # primero el marco que usa el navegador; luego Navegar_6 (el que funcionó en 2007–2026) y los demás
        candidatos = ([urljoin(f"{self.base}/", src), f"{self.base}/Navegar_6.aspx?y={anio}", f"{self.base}/Navegar.aspx?y={anio}"]
                      + [f"{self.base}/Navegar_{n}.aspx?y={anio}" for n in range(1, 8)])
        vistos, pg = [], None
        for url in candidatos:
            if url in vistos:
                continue
            vistos.append(url)
            pg, err = self._abrir_raiz(anio, url)
            if pg is not None:
                break
            self.log(f"   · {anio}: {url.rsplit('/', 1)[-1]} → {err}")
        if pg is None:
            return False
        m = re.search(r"Navegar(_\d+)?\.aspx", pg.url or url)          # tras una redirección vale la dirección final
        self.pagina = (m.group(1) or "").lstrip("_") if m else "?"
        self._cache[()] = pg
        self._anio = anio
        self.log(f"🔎 {anio}: Navegar{'_' + self.pagina if self.pagina else ''}.aspx · raíz con {len(pg.filas)} fuentes")
        return True

    def _abrir_raiz(self, anio, url):
        """GET del marco y, si muestra solo la fila TOTAL, pulsa «Fuente» (como en la página). Devuelve (página de fuentes, error)."""
        pg, err = self._pedir("GET", url, reintentos=2)
        if pg is None:
            return None, err
        if pg.anio and str(pg.anio) != str(anio):
            return None, f"la página quedó en el año {pg.anio}"
        if not pg.tiene_tabla and not pg.boton(D_FUENTE):
            return None, f"sin tabla («{pg.texto[:120]}»; botones: {', '.join(pg.botones) or '–'})"
        if not misma_dim(pg.dim, D_FUENTE) and pg.boton(D_FUENTE):
            # la raíz muestra solo TOTAL (o ninguna fila): se pulsa «Fuente», como en la página
            pf, err = self._postear(pg, pg.filas[0][0] if pg.filas else None, D_FUENTE, validar=False)
            if pf is None:
                return None, f"al pulsar «Fuente»: {err}"
            pg = pf
        if not misma_dim(pg.dim, D_FUENTE):
            return None, (f"la raíz lista «{pg.dim}» (filas: {', '.join(e for _, e, _ in pg.filas[:5]) or '–'}; "
                          f"botones: {', '.join(pg.botones)})")
        return pg, None

    def _postear(self, padre, cod, dim, validar=True):
        """Desde la página `padre`: elige la fila `cod` y pulsa el botón de `dim`. Valida historial y cabecera."""
        if cod is not None and cod not in padre.radios:
            return None, f"la fila «{cod}» no está en «{padre.dim}»"
        b = padre.boton(dim)
        if b is None:
            return None, f"sin botón «{dim}» (botones: {', '.join(padre.botones) or '–'})"
        datos = dict(padre.ocultos)
        datos.update({"__EVENTTARGET": "", "__EVENTARGUMENT": "", b[1]: b[0]})
        if cod is not None:
            datos["grp1"] = padre.radios[cod]
        pg, err = self._pedir("POST", urljoin(padre.url, padre.accion) if padre.accion else padre.url, datos)
        if pg is None:
            return None, err
        if not pg.tiene_tabla:
            return None, (f"SIN_TABLA (año de la página: {pg.anio or '?'}; botones: {', '.join(pg.botones) or '–'}; "
                          f"la página respondió: «{pg.texto[:160]}»)")
        etq = next((e for c, e, _ in padre.filas if c == cod), cod)
        if not misma_dim(pg.dim, dim):
            return None, f"se pidió «{dim}» y la tabla lista «{pg.dim}»"
        if validar and (len(pg.filtros) != len(padre.filtros) + 1 or solo_letras(etq) not in solo_letras(pg.filtros[-1][0])):
            return None, f"ESTADO: filtro no aplicado (historial: {' › '.join(f for f, _ in pg.filtros)})"
        return pg, None

    def listar(self, anio, ruta, dim):
        """Página que lista `dim` bajo los filtros `ruta` = [(dimensión, código)…] (la primera dimensión es la de la raíz)."""
        if not self.preparar(anio):
            return None, "no se pudo abrir el año"
        if not ruta:
            pg = self._cache[()]
            return (pg, None) if misma_dim(pg.dim, dim) else (None, f"la raíz lista «{pg.dim}»")
        dims = [d for d, _ in ruta[1:]] + [dim]          # la página tras el filtro i lista dims[i]
        err = ""
        for intento in (1, 2):
            pg = self._cache[()]
            for i in range(len(ruta)):
                clave = tuple(ruta[:i + 1]) + (dims[i],)  # filtros hasta i + la dimensión que lista esa página
                if clave in self._cache:
                    pg = self._cache[clave]; continue
                nueva, err = self._postear(pg, ruta[i][1], dims[i])
                if nueva is None:
                    break
                self._cache[clave] = pg = nueva
            else:
                return pg, None
            # una página guardada ya no sirve (el estado quedó en la sesión): se rehace el camino desde la raíz
            if intento == 1 and err.startswith("ESTADO") and self.preparar(anio, forzar=True):
                continue
            break
        return None, err

    @staticmethod
    def elegir(pg, patron):
        return next(((c, e, v) for c, e, v in pg.filas if re.search(patron, e, re.I)), None)

    # ── recorrido del canon ──
    def ruta_tipos(self, anio):
        """Esquema del año: desde 2007 fuente 5 → rubro 18; hasta 2006 fuente 01 (Canon y sobrecanon). Devuelve (ruta, tipos)."""
        pg, err = self.listar(anio, [], D_FUENTE)
        if err:
            return None, [], err
        for patron, con_rubro in ((r"^0?5:\s*RECURSOS DETERMINADOS", True), (r"^0?1:\s*CANON\b", False)):
            f = self.elegir(pg, patron)
            if not f:
                continue
            ruta = [(D_FUENTE, f[0])]
            if con_rubro:
                pr, err = self.listar(anio, ruta, D_RUBRO)
                if err:
                    return None, [], f"rubro: {err}"
                r18 = self.elegir(pr, r"^18:")
                if not r18:
                    return None, [], "sin rubro 18"
                ruta = ruta + [(D_RUBRO, r18[0])]
            pt, err = self.listar(anio, ruta, D_TIPO)
            if err:
                return None, [], f"tipo de recurso: {err}"
            return ruta, pt.filas, None
        return None, [], "sin fuente 5 (Recursos determinados) ni 01 (Canon y sobrecanon)"

    def niveles(self, anio, ruta, tipo):
        """Distribución por nivel de gobierno (E nacional, M locales, R regionales) de un tipo de recurso.
        Devuelve (ruta con el tipo, página «Nivel de Gobierno», error)."""
        r = ruta + [(D_TIPO, tipo[0])]
        pn, err = self.listar(anio, r, D_NIVEL)
        return r, pn, (f"nivel de gobierno: {err}" if err else None)

    def sectores_pliegos(self, anio, ruta, tipo):
        """Cuando la consulta no ofrece «Nivel de Gobierno» (2003): reparto del tipo por sector y pliego.
        Devuelve ([(sector, pliego, recaudado)…], error)."""
        r = ruta + [(D_TIPO, tipo[0])]
        ps, err = self.listar(anio, r, D_SECTOR)
        if err:
            return None, f"sector: {err}"
        salida = []
        for c, et, v in ps.filas:
            pp, err = self.listar(anio, r + [(D_SECTOR, c)], D_PLIEGO)
            if err:
                salida.append((et, "", v)); continue
            salida += [(et, ep, vp) for _, ep, vp in pp.filas]
        return salida, None

    def departamentos_de_nivel(self, anio, r_tipo, nivel):
        """Departamentos bajo un nivel de gobierno (si la página no ofrece «Departamento» pero sí «Gob.Loc./Mancom.»,
        pasa antes por la fila M de gobiernos locales, como en los años recientes)."""
        pn, err = self.listar(anio, r_tipo, D_NIVEL)
        if err:
            return None, None, f"nivel: {err}"
        r = r_tipo + [(D_NIVEL, nivel[0])]
        if nivel[0] == "M" and pn.boton(D_DEPTO) is None and pn.boton(D_GOBLOC) is not None:
            pg, err = self.listar(anio, r, D_GOBLOC)
            if err:
                return None, None, f"gob. local: {err} · botones bajo «Nivel de Gobierno»: {', '.join(pn.botones)}"
            g = self.elegir(pg, r"^M:") or (pg.filas[0] if pg.filas else None)
            if g is None:
                return None, None, "gob. local sin filas"
            r = r + [(D_GOBLOC, g[0])]
        if nivel[0] == "R" and pn.boton(D_DEPTO) is None and pn.boton(D_SECTOR) is not None:
            return self._regionales_por_pliego(anio, r)
        pd_, err = self.listar(anio, r, D_DEPTO)
        if err and nivel[0] == "M":
            err += f" · botones bajo «Nivel de Gobierno»: {', '.join(pn.botones)}"
        return r, pd_, (f"departamento: {err}" if err else None)

    def _regionales_por_pliego(self, anio, r):
        """Años recientes: bajo «Gobiernos Regionales» no hay botón «Departamento»; se baja por Sector → Pliego
        («GOBIERNO REGIONAL DEL DEPARTAMENTO DE LORETO») y cada pliego se asigna al departamento que nombra.
        Devuelve (ruta, objeto con .filas [(código, «NN: DEPARTAMENTO», recaudado)], error)."""
        ps, err = self.listar(anio, r, D_SECTOR)
        if err:
            return r, None, f"departamento (por sector): {err}"
        suma, sin = {}, []
        for c, et, v in ps.filas:
            pp, err = self.listar(anio, r + [(D_SECTOR, c)], D_PLIEGO)
            if err:
                sin.append(f"{et}: {err}"); continue
            for _, ep, vp in pp.filas:
                dep = depto_en(ep)
                if dep:
                    suma[dep] = suma.get(dep, 0.0) + vp
                elif vp:
                    sin.append(ep)
        if not suma:
            return r, None, "departamento (por pliego): ningún pliego nombra un departamento" + (f" ({'; '.join(sin[:3])})" if sin else "")
        class _Dep:
            pass
        out = _Dep()
        out.filas = [(DEPTOS[d], f"{DEPTOS[d]}: {d}", v) for d, v in sorted(suma.items())]
        out.sin_asignar = sin
        return r, out, None

    def hasta_departamentos(self, anio, ruta, tipo):
        """Agrega tipo de recurso y nivel M (y gob. local si corresponde) y lista los departamentos."""
        r, pn, err = self.niveles(anio, ruta, tipo)
        if err:
            return None, None, err
        m = self.elegir(pn, r"^M:")
        if not m:
            return None, None, "sin gobiernos locales (nivel M) en este tipo de recurso"
        r, pd_, err = self.departamentos_de_nivel(anio, r, m)
        if err:
            return None, None, err
        return r, pd_, None

    def municipalidades(self, anio, ruta_dep, dep):
        """Municipalidades del departamento elegido; total = historial del departamento."""
        pm, err = self.listar(anio, ruta_dep + [(D_DEPTO, dep[0])], D_MUNI)
        if err:
            return None, None, f"municipalidad: {err}"
        return pm.filas, (pm.filtros[-1][1] if pm.filtros else None), None

DEPTOS = {"AMAZONAS": "01", "ANCASH": "02", "APURIMAC": "03", "AREQUIPA": "04", "AYACUCHO": "05", "CAJAMARCA": "06",
          "CALLAO": "07", "CUSCO": "08", "HUANCAVELICA": "09", "HUANUCO": "10", "ICA": "11", "JUNIN": "12", "LA LIBERTAD": "13",
          "LAMBAYEQUE": "14", "LIMA": "15", "LORETO": "16", "MADRE DE DIOS": "17", "MOQUEGUA": "18", "PASCO": "19", "PIURA": "20",
          "PUNO": "21", "SAN MARTIN": "22", "TACNA": "23", "TUMBES": "24", "UCAYALI": "25"}

def depto_en(etiqueta):
    """Departamento que nombra un pliego («453: GOBIERNO REGIONAL DEL DEPARTAMENTO DE LORETO» → LORETO)."""
    t = " " + solo_letras(etiqueta).upper() + " "
    hall = [d for d in DEPTOS if f" {d} " in t]
    hall = [d for d in hall if not any(d != o and d in o for o in hall)]   # «LIMA» dentro de otro nombre no cuenta
    return hall[0] if len(hall) == 1 else None

def nombre_municipalidad(etiqueta):
    """«01-301422: MUNICIPALIDAD PROVINCIAL DE MAYNAS - IQUITOS» → (código, nivel, municipalidad, provincia)."""
    cod, _, txt = etiqueta.partition(":")
    txt = txt.strip().upper()
    m = re.match(r"MUNICIPALIDAD\s+(PROVINCIAL|DISTRITAL)\s+(?:(?:DE\s+LA|DEL|DE)\s+)?(.*)$", txt)
    if not m:
        return cod.strip(), "", txt, ""
    nivel, nom = m.group(1), m.group(2).strip()
    if nivel == "PROVINCIAL":
        prov, _, capital = nom.partition(" - ")
        return cod.strip(), nivel, (capital or prov).strip(), prov.strip()
    return cod.strip(), nivel, nom, ""


def norm_may(t):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()).strip().upper()
# departamentos con canon petrolero (los de la lista); «LOS 6 PETROLEROS» los recorre todos
PETROLEROS = {"16": "LORETO", "25": "UCAYALI", "20": "PIURA", "24": "TUMBES", "10": "HUANUCO", "08": "CUSCO"}
TODOS = DEPARTAMENTO.strip().upper() in ("LOS 6 PETROLEROS", "TODOS")
# referencias de PERUPETRO (si existen en DATA): total municipal por departamento y año, y provincia de cada municipalidad
dist_pp = bajar("PP_canon_distribucion_mes.csv", datos_drive)
mun_pp = bajar("PP_canon_municipal_mes.csv", datos_drive)
pp_mun = (dist_pp[dist_pp["TIPO_BENEFICIARIO"] == "MUNICIPALIDADES"].assign(_D=dist_pp["DEPARTAMENTO"].map(norm_may))
          .groupby(["_D", "AÑO"])["MONTO_SOLES"].sum().to_dict() if dist_pp is not None else {})
prov_de = ({norm_may(m): norm_may(p) for m, p in zip(mun_pp["MUNICIPALIDAD"], mun_pp["PROVINCIA"]) if isinstance(p, str)}
           if mun_pp is not None else {})

if not (2003 <= ANIO_INICIO <= ANIO_FIN <= date.today().year + 1):
    raise SystemExit(f"❌ Revisa los años: ANIO_INICIO = {ANIO_INICIO}, ANIO_FIN = {ANIO_FIN}. "
                     f"Deben ir de 2003 a {date.today().year} y el inicio no puede ser mayor que el fin.")
m = MEFIngresos(log=print)
if not m.preparar(min(ANIO_FIN, date.today().year)):
    raise SystemExit("❌ No se pudo abrir la consulta (ver el mensaje de arriba): el MEF cambió la página o bloqueó la consulta.")
filas, ctrl, niv_filas, ctrl_niv = [], [], [], []
MENSUAL = DETALLE.endswith("mensual")
filas_mes, niv_mes, ctrl_mes = [], [], []          # versión mensual: municipalidades, niveles y control (suma de meses = año)

def meses_de(anio, ruta):
    """Meses (código, número, etiqueta) bajo la página de `ruta` (botón «Mes»)."""
    pm, e = m.listar(anio, ruta, D_MES)
    if e:
        return None, e
    return [(c, mes_num(c, et), et) for c, et, _ in pm.filas], None

def registrar_niveles(anio, ruta, t, normal=False):
    """Distribución del tipo de recurso por nivel de gobierno (E, M, R): total país y por departamento."""
    etq_t = t[1] + (" (todos los canones)" if normal else "")
    fuente = FUENTE + (" · tipo 0 NORMAL: todos los canones, no solo el petrolero" if normal else "")
    r_t, pn, e = m.niveles(anio, ruta, t)
    if e and "sin botón" in e:
        # 2003: la consulta no tiene «Nivel de Gobierno»; se registra el reparto por sector y pliego (p. ej. todo al CND)
        sp, e2 = m.sectores_pliegos(anio, ruta, t)
        if sp is None:
            ctrl_niv.append({"AÑO": anio, "TIPO_RECURSO": etq_t, "ESTADO": f"ERROR: {e} · {e2}"}); return
        for sec, pli, v in sp:
            niv_filas.append({"AÑO": anio, "TIPO_RECURSO": etq_t, "NIVEL_GOBIERNO": "SIN NIVEL EN LA CONSULTA · " + " › ".join(x for x in (sec, pli) if x),
                              "DEPARTAMENTO": "TOTAL PAÍS", "RECAUDADO_SOLES": round(v, 2), "FUENTE": fuente})
        suma = sum(v for _, _, v in sp)
        ctrl_niv.append({"AÑO": anio, "TIPO_RECURSO": etq_t, "NIVELES": " | ".join(f"{sec} › {pli}: S/ {v:,.0f}" for sec, pli, v in sp),
                         "SUMA_NIVELES": round(suma, 2), "TOTAL_TIPO": round(t[2], 2), "DIF": round(suma - t[2], 2),
                         "SIN_DETALLE_DEPARTAMENTO": "la consulta de este año no tiene «Nivel de Gobierno»: reparto por sector y pliego",
                         "ESTADO": "OK" if abs(suma - t[2]) <= max(1, len(sp)) else "REVISAR"})
        print(f"   🏛️ {anio} · {etq_t} (sin nivel de gobierno): " + " · ".join(f"{(pli or sec).split(':', 1)[-1].strip().title()} S/ {v/1e6:,.2f} millones" for sec, pli, v in sp))
        return
    if e:
        ctrl_niv.append({"AÑO": anio, "TIPO_RECURSO": etq_t, "ESTADO": f"ERROR: {e}"}); return
    suma = sum(v for _, _, v in pn.filas)
    sin_dep = []
    for c, et, v in pn.filas:
        niv_filas.append({"AÑO": anio, "TIPO_RECURSO": etq_t, "NIVEL_GOBIERNO": et, "DEPARTAMENTO": "TOTAL PAÍS",
                          "RECAUDADO_SOLES": round(v, 2), "FUENTE": fuente})
        _, pdep, e2 = m.departamentos_de_nivel(anio, r_t, (c, et, v))
        if pdep is None:
            sin_dep.append(f"{et}: {e2}"); continue
        for cd, ed, vd in pdep.filas:
            niv_filas.append({"AÑO": anio, "TIPO_RECURSO": etq_t, "NIVEL_GOBIERNO": et,
                              "DEPARTAMENTO": norm_may(ed.split(":", 1)[1] if ":" in ed else ed),
                              "RECAUDADO_SOLES": round(vd, 2), "FUENTE": fuente})
    ctrl_niv.append({"AÑO": anio, "TIPO_RECURSO": etq_t, "NIVELES": " | ".join(f"{et}: S/ {v:,.0f}" for _, et, v in pn.filas),
                     "SUMA_NIVELES": round(suma, 2), "TOTAL_TIPO": round(t[2], 2), "DIF": round(suma - t[2], 2),
                     "SIN_DETALLE_DEPARTAMENTO": "; ".join(sin_dep),
                     "ESTADO": "OK" if abs(suma - t[2]) <= max(1, len(pn.filas)) else "REVISAR"})
    print(f"   🏛️ {anio} · {etq_t}: " + " · ".join(f"{et.split(':', 1)[-1].strip().title()} S/ {v/1e6:,.2f} millones" for _, et, v in pn.filas))
    if MENSUAL:
        ms, e = meses_de(anio, r_t)
        if ms is None:
            ctrl_mes.append({"AÑO": anio, "DEPARTAMENTO": "TOTAL PAÍS", "TIPO_RECURSO": etq_t, "ALCANCE": "NIVEL DE GOBIERNO",
                             "ESTADO": f"ERROR meses: {e}"}); return
        suma_m = 0.0
        for mc, mn, met in ms:
            pnm, e = m.listar(anio, r_t + [(D_MES, mc)], D_NIVEL)
            if e:
                ctrl_mes.append({"AÑO": anio, "MES": mn, "DEPARTAMENTO": "TOTAL PAÍS", "TIPO_RECURSO": etq_t,
                                 "ALCANCE": "NIVEL DE GOBIERNO", "ESTADO": f"ERROR: {e}"}); continue
            for c, et, v in pnm.filas:
                niv_mes.append({"AÑO": anio, "MES": mn, "TIPO_RECURSO": etq_t, "NIVEL_GOBIERNO": et, "DEPARTAMENTO": "TOTAL PAÍS",
                                "RECAUDADO_SOLES": round(v, 2), "FUENTE": fuente})
                suma_m += v
        ctrl_mes.append({"AÑO": anio, "DEPARTAMENTO": "TOTAL PAÍS", "TIPO_RECURSO": etq_t, "ALCANCE": "NIVEL DE GOBIERNO",
                         "MESES": len(ms), "SUMA_MESES": round(suma_m, 2), "TOTAL_ANUAL": round(suma, 2), "DIF": round(suma_m - suma, 2),
                         "ESTADO": "OK" if abs(suma_m - suma) <= max(1, len(ms) * len(pn.filas)) else "REVISAR"})

for anio in range(ANIO_INICIO, ANIO_FIN + 1):
    etq_dep = "LOS 6 PETROLEROS" if TODOS else norm_may(DEPARTAMENTO.split(":", 1)[1])
    ruta, tipos, err = m.ruta_tipos(anio)
    if err:
        print(f"⚠️ {anio}: {err}"); ctrl.append({"AÑO": anio, "DEPARTAMENTO": etq_dep, "ESTADO": f"ERROR: {err}"}); continue
    resultados = []                                     # [(tipo, ruta hasta departamento, página de departamentos)]
    motivos = []
    for t in [t for t in tipos if re.search("PETROLER", t[1], re.I)]:   # 1) distribución por nivel de gobierno
        registrar_niveles(anio, ruta, t)
    for t in [t for t in tipos if re.search("PETROLER", t[1], re.I)]:   # 2) municipalidades (nivel M)
        r, pdep, e = m.hasta_departamentos(anio, ruta, t)
        if r: resultados.append((t, r, pdep, False))
        else: motivos.append(f"{t[1]}: {e}")
    usa_normal = False
    if SI_NO_HAY_PETROLERO.startswith("Usar") and anio < 2007:
        # 2005–2006: el canon municipal está sobre todo en «0: NORMAL» (todos los canones); el tipo 4 petrolero trae poco
        # (Loreto 2005: S/ 0,91 millones sin municipalidades listadas; 2006: S/ 0,14 millones). Se suman ambos tipos, cada
        # fila con su rótulo. Desde 2007 NO se usa «0: CANON»:
        # en 2008 y 2009 ese tipo da a los gobiernos locales de Loreto S/ 1,18 y S/ 0,19 millones (capturas del usuario),
        # mientras el canon petrolero va por los tipos 5 y K.
        t0 = next((t for t in tipos if re.match(r"^0:\s*NORMAL\s*$", t[1], re.I)), None)
        if t0:
            r, pdep, e = m.hasta_departamentos(anio, ruta, t0)
            if r:
                resultados.append((t0, r, pdep, True)); usa_normal = True
                registrar_niveles(anio, ruta, t0, normal=True)
            else: motivos.append(f"{t0[1]}: {e}")
    if not resultados:
        msg = "SIN DETALLE MUNICIPAL (" + "; ".join(motivos or ["sin tipo de recurso petrolero"]) + ")"
        print(f"ℹ️ {anio}: {msg}"); ctrl.append({"AÑO": anio, "DEPARTAMENTO": etq_dep, "ESTADO": msg}); continue
    if TODOS:
        deps = sorted({(c, e) for _, _, pdep, _ in resultados for c, e, v in pdep.filas
                       if (v or 0) > 0 and c.zfill(2) in PETROLEROS})
    else:
        cod = DEPARTAMENTO.split(":")[0].strip()
        deps = [(cod, DEPARTAMENTO)]
    for cod_d, etq in deps:
        DEP = norm_may(etq.split(":", 1)[1] if ":" in etq else etq)
        suma = tot = 0.0; n = 0; detalle = []
        for t, r, pdep, normal in resultados:
            dep = next(((c, e, v) for c, e, v in pdep.filas if c.lstrip("0") == cod_d.lstrip("0")), None)
            if dep is None:
                detalle.append(f"{t[1]}: sin {DEP}"); continue
            fs, total, e = m.municipalidades(anio, r, dep)
            if fs is None:
                detalle.append(f"{t[1]}: {e}"); continue
            for c, et, v in fs:                         # todas las filas: en 2005 los nombres no siempre dicen «MUNICIPALIDAD»
                codigo, nivel, mu, prov = nombre_municipalidad(et)
                filas.append({"AÑO": anio, "DEPARTAMENTO": DEP, "CODIGO": codigo, "MUNICIPALIDAD_MEF": et.split(":", 1)[1].strip(),
                              "MUNICIPALIDAD": mu, "NIVEL": nivel, "PROVINCIA": prov or prov_de.get(norm_may(mu), ""),
                              "TIPO_RECURSO": t[1] + (" (todos los canones)" if normal else ""), "RECAUDADO_SOLES": round(v, 2),
                              "FUENTE": FUENTE + (" · tipo 0 NORMAL: todos los canones, no solo el petrolero" if normal else "")})
                suma += v; n += 1
            tot += total or 0.0
            if MENSUAL:
                anual = sum(v for _, _, v in fs)
                ms, e = meses_de(anio, r)                # meses bajo la misma página que lista los departamentos
                if ms is None:
                    ctrl_mes.append({"AÑO": anio, "DEPARTAMENTO": DEP, "TIPO_RECURSO": t[1], "ALCANCE": "MUNICIPALIDADES",
                                     "ESTADO": f"ERROR meses: {e}"})
                else:
                    suma_m = 0.0; errores = []
                    for mc, mn, met in ms:
                        pdm, e = m.listar(anio, r + [(D_MES, mc)], D_DEPTO)
                        dm = None if e else next(((c, et, v) for c, et, v in pdm.filas if c.lstrip("0") == cod_d.lstrip("0")), None)
                        if dm is None:
                            if e: errores.append(f"{met}: {e}")
                            continue                     # ese mes no hubo recaudación en el departamento
                        pmu, e = m.listar(anio, r + [(D_MES, mc), (D_DEPTO, dm[0])], D_MUNI)
                        if e:
                            errores.append(f"{met}: {e}"); continue
                        for c, et, v in pmu.filas:
                            codigo, nivel, mu, prov = nombre_municipalidad(et)
                            filas_mes.append({"AÑO": anio, "MES": mn, "DEPARTAMENTO": DEP, "CODIGO": codigo,
                                              "MUNICIPALIDAD_MEF": et.split(":", 1)[1].strip(), "MUNICIPALIDAD": mu, "NIVEL": nivel,
                                              "PROVINCIA": prov or prov_de.get(norm_may(mu), ""),
                                              "TIPO_RECURSO": t[1] + (" (todos los canones)" if normal else ""), "RECAUDADO_SOLES": round(v, 2),
                                              "FUENTE": FUENTE + (" · tipo 0 NORMAL: todos los canones, no solo el petrolero" if normal else "")})
                            suma_m += v
                    ctrl_mes.append({"AÑO": anio, "DEPARTAMENTO": DEP, "TIPO_RECURSO": t[1] + (" (todos los canones)" if normal else ""),
                                     "ALCANCE": "MUNICIPALIDADES", "MESES": len(ms), "SUMA_MESES": round(suma_m, 2),
                                     "TOTAL_ANUAL": round(anual, 2), "DIF": round(suma_m - anual, 2), "ERRORES": "; ".join(errores),
                                     "ESTADO": "OK" if not errores and abs(suma_m - anual) <= max(1, len(fs)) else "REVISAR"})
                    print(f"   🗓️ {anio} · {DEP} · {t[1]}: {len(ms)} meses · suma S/ {suma_m/1e6:,.2f} millones vs año S/ {anual/1e6:,.2f} millones")
            detalle.append(f"{t[1]}{' (todos los canones)' if normal else ''}: S/ {total or 0:,.0f}")
        ref = pp_mun.get((DEP, anio))
        ctrl.append({"AÑO": anio, "DEPARTAMENTO": DEP, "TIPOS": " | ".join(detalle), "MUNICIPALIDADES": n,
                     "SUMA_MUNICIPALIDADES": round(suma, 2), "TOTAL_DEPARTAMENTO_MEF": round(tot, 2), "DIF": round(suma - tot, 2),
                     "PERUPETRO_MUNICIPAL": round(ref, 2) if ref else None, "MEF_SOBRE_PERUPETRO": round(suma / ref, 4) if ref else None,
                     "ESTADO": ("OK" if n and abs(suma - tot) <= max(1, n) else "REVISAR" if n else "SIN MUNICIPALIDADES")})
        print(f"{'✅' if n else 'ℹ️'} {anio} · {DEP}: {n} filas de municipalidades · S/ {suma/1e6:,.2f} millones"
              + (" (incluye NORMAL: todos los canones)" if usa_normal else "") + " [" + " | ".join(detalle) + "]" + (f" · PERUPETRO S/ {ref/1e6:,.2f} millones" if ref else ""))
print(f"Última página usada: Navegar{'_' + m.pagina if m.pagina else ''}.aspx")

# ── Distribución por nivel de gobierno (se acumula por año) ──
nv = pd.DataFrame(niv_filas)
previo_nv = bajar("PP_canon_nivel_gobierno_mef_anio.csv", datos_drive)
if previo_nv is not None and len(nv):
    nv = pd.concat([previo_nv[~previo_nv["AÑO"].isin(set(nv["AÑO"]))], nv], ignore_index=True)
elif previo_nv is not None:
    nv = previo_nv
if len(nv):
    nv = nv.sort_values(["AÑO", "TIPO_RECURSO", "NIVEL_GOBIERNO", "DEPARTAMENTO"],
                        key=lambda s: s.map(lambda x: ("0" if x == "TOTAL PAÍS" else "1") + str(x)) if s.name == "DEPARTAMENTO" else s)
    p = os.path.join(ruta_local, "PP_canon_nivel_gobierno_mef_anio.csv"); nv.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_ID, datos_drive)
cn = pd.DataFrame(ctrl_niv)
if len(cn):
    cn.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
    previo_cn = bajar("CONTROL PP canon_nivel_gobierno_mef.csv", control_drive)
    if previo_cn is not None:
        cn = pd.concat([previo_cn[~previo_cn["AÑO"].isin(set(cn["AÑO"]))], cn], ignore_index=True).sort_values(["AÑO", "TIPO_RECURSO"])
    p = os.path.join(ruta_local, "CONTROL PP canon_nivel_gobierno_mef.csv"); cn.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_CONTROL_ID, control_drive)
    print(f"☁️ DATA/PP_canon_nivel_gobierno_mef_anio.csv ({len(nv)} filas) · CONTROL/CONTROL PP canon_nivel_gobierno_mef.csv")
    display(cn[cn["AÑO"].between(ANIO_INICIO, ANIO_FIN)].drop(columns=["FECHA_EXTRACCION"]))

# ── Versión mensual (se acumula: cada corrida reemplaza los años que vuelve a leer) ──
def acumular(df, nombre, carpeta, existentes, claves, orden):
    previo = bajar(nombre, existentes)
    if previo is not None and len(df):
        leidos = set(map(tuple, df[claves].drop_duplicates().values.tolist()))
        previo = previo[[tuple(x) not in leidos for x in previo[claves].values.tolist()]]
        df = pd.concat([previo, df], ignore_index=True)
    elif previo is not None:
        df = previo
    if len(df):
        df = df.sort_values(orden)
        p = os.path.join(ruta_local, nombre); df.to_csv(p, index=False, encoding="utf-8-sig")
        subir(p, carpeta, existentes)
    return df
if MENSUAL:
    fm = acumular(pd.DataFrame(filas_mes), "PP_canon_municipal_mef_mes.csv", DRIVE_FOLDER_ID, datos_drive,
                  ["DEPARTAMENTO", "AÑO"], ["DEPARTAMENTO", "AÑO", "MES", "MUNICIPALIDAD", "TIPO_RECURSO"])
    nm = acumular(pd.DataFrame(niv_mes), "PP_canon_nivel_gobierno_mef_mes.csv", DRIVE_FOLDER_ID, datos_drive,
                  ["AÑO"], ["AÑO", "MES", "TIPO_RECURSO", "NIVEL_GOBIERNO"])
    cm = pd.DataFrame(ctrl_mes)
    if len(cm):
        cm.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
        cm = acumular(cm, "CONTROL PP canon_mef_mes.csv", DRIVE_FOLDER_CONTROL_ID, control_drive,
                      ["DEPARTAMENTO", "AÑO"], ["AÑO", "DEPARTAMENTO", "ALCANCE", "TIPO_RECURSO"])
        print(f"☁️ DATA/PP_canon_municipal_mef_mes.csv ({len(fm)} filas) · DATA/PP_canon_nivel_gobierno_mef_mes.csv ({len(nm)} filas) · "
              f"CONTROL/CONTROL PP canon_mef_mes.csv")
        display(cm[cm["AÑO"].between(ANIO_INICIO, ANIO_FIN)].drop(columns=["FECHA_EXTRACCION"]))

# ── Guardar (se acumula: cada corrida reemplaza los departamentos y años que vuelve a leer) ──
nuevo = pd.DataFrame(filas)
_pet = set(PETROLEROS.values())
previo = bajar("PP_canon_municipal_mef_anio.csv", datos_drive)
if previo is not None and len(nuevo):
    leidos = set(zip(nuevo["DEPARTAMENTO"], nuevo["AÑO"]))
    previo = previo[[(d, a) not in leidos for d, a in zip(previo["DEPARTAMENTO"], previo["AÑO"])]]
    nuevo = pd.concat([previo, nuevo], ignore_index=True)
elif previo is not None:
    nuevo = previo
if len(nuevo):
    nuevo = nuevo[nuevo["DEPARTAMENTO"].isin(_pet)]           # solo los departamentos con canon petrolero
    nuevo = nuevo.sort_values(["DEPARTAMENTO", "AÑO", "NIVEL", "MUNICIPALIDAD", "TIPO_RECURSO"], ascending=[True, True, False, True, True])
    p = os.path.join(ruta_local, "PP_canon_municipal_mef_anio.csv"); nuevo.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_ID, datos_drive)
c = pd.DataFrame(ctrl, columns=["AÑO", "DEPARTAMENTO", "ESTADO"] if not ctrl else None); c.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
previo_c = bajar("CONTROL PP canon_municipal_mef.csv", control_drive)
if previo_c is not None:              # filas de versiones anteriores del notebook («16: LORETO», «NO INTENTADO…») se descartan
    previo_c = previo_c[~previo_c["DEPARTAMENTO"].astype(str).str.contains(":") & ~previo_c["ESTADO"].astype(str).str.startswith("NO INTENTADO")]
if previo_c is not None:
    leidos_c = set(zip(c["DEPARTAMENTO"], c["AÑO"]))
    c = pd.concat([previo_c[[(d, a) not in leidos_c for d, a in zip(previo_c["DEPARTAMENTO"], previo_c["AÑO"])]], c],
                  ignore_index=True).sort_values(["AÑO", "DEPARTAMENTO"])
c = c[c["DEPARTAMENTO"].isin(_pet | {"LOS 6 PETROLEROS", "TODOS"})]
p = os.path.join(ruta_local, "CONTROL PP canon_municipal_mef.csv"); c.to_csv(p, index=False, encoding="utf-8-sig")
subir(p, DRIVE_FOLDER_CONTROL_ID, control_drive)
print(f"\n☁️ DATA/PP_canon_municipal_mef_anio.csv ({len(nuevo)} filas) · CONTROL/CONTROL PP canon_municipal_mef.csv")
display(c[c["AÑO"].between(ANIO_INICIO, ANIO_FIN)].drop(columns=["FECHA_EXTRACCION"]))
